[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barmag/fanous-llm-lens/blob/main/notebooks/education/stage2_dash2_composition_induction_reference.ipynb)

<div dir="rtl">

# المرحلة ٢داش²: طبقتين انتباه — التركيب ورأس الاستقراء (على عربي حقيقي)

ده أكتف نوتبوك في المنهج. بيفترض إنك خلصت **٢ج** (حدس رأس الاستقراء على نموذج صغير) و**٢داش** (طريقة تفكيك الدائرة بإيدك).

**ملاحظة أمانة:** النموذج ده **من غير LayerNorm + shortformer** عشان التفكيك يطلع **بالظبط**؛ ده اختيار مقصود، مش نفس معمار الورقة بالحرف. بنعيد إنتاج **نتايج** الورقة (الاستقراء عن طريق التركيب)، مش المعمار الحرفي. "نفس المقياس" ≠ "نفس المعمار".

</div>

# Stage 2dash²: two attention layers — composition & the induction head (on real Arabic)

The densest notebook in the curriculum. Assumes you have done **2c** (induction intuition on a toy model) and **2dash** (the by-hand circuit-decomposition method).

**Faithfulness note:** this model is **LN-free + shortformer** so the decomposition is *exact* — a deliberate choice, **not the paper's literal architecture**. We reproduce the paper's *results* (induction via composition), not its literal model. "Faithful-scale" ≠ "faithful architecture".

In [ ]:
# Setup: install deps + fetch the shared helper on Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q transformer_lens tokenizers huggingface_hub plotly
    !pip uninstall -y -q torchaudio
    !wget -q https://raw.githubusercontent.com/barmag/fanous-llm-lens/main/notebooks/education/tiny.py
import tiny
import torch
import plotly.graph_objects as go

torch.manual_seed(42)
print("device:", tiny.device())

<div dir="rtl">

## ١. مقدمة: ليه طبقة واحدة مش كفاية · 1. Why one layer isn't enough

موديل الـ **٢داش** بيتفكّك بالظبط لـ bigram + دوائر skip-trigram. ده قوي — لكنه عنده حد واضح:

**الـ skip-trigram بيقدر ينسخ توكن قديم، لكن بس لو التوكن ده ظهر في السياق التدريبي مرتبط بوجهة معينة.** لما تعدّي توكن جديد أو نادر ما شافهوش الـ QK و OV سوا في التدريب، الرأس مش عارف يعمل إيه.

بس في حالة مهمة بتكسر التوقع: **التتابع المتكرر** — الجزء الأول يكون عشوائي، الجزء الثاني نسخة منه. ده بالظبط اللي بيتعلمه **رأس الاستقراء**:

> رأس الاستقراء بيبص على التوكن الحالي، بيدور على آخر مرة ظهر فيها في السياق، وبيتنبأ بالتوكن اللي ماشي بعده.

الطبقة الواحدة **مش قادرة تعمل ده** لوحدها: محتاج طبقتين — الأولى تتتبع "شفت ده قبل كده" والتانية تنسخ "إيه اللي ماشي بعده".

</div>

## 1. Why one layer isn't enough

The **2dash** model decomposes exactly into a bigram + skip-trigram ensemble. That's powerful — but it has a clear limit:

**A skip-trigram can copy an old token, but only if that token appeared in training paired with a known destination in the QK and OV matrices.** On a novel or rare token the head has never seen at this position, it has no signal.

There's one important failure case that breaks the expectation: **a repeated sequence** — the first half is arbitrary, the second half repeats it. This is precisely what an **induction head** learns to do:

> An induction head looks at the current token, finds the last time it appeared in context, and predicts the token that followed it.

A single layer **cannot do this alone**: it requires two layers — the first to track "I've seen this before" and the second to copy "what came next".

In [ ]:
import os

CKPT1_DIR = os.path.join(os.path.dirname(os.path.abspath(tiny.__file__)), "checkpoints", "stage2dash")
CKPT2_DIR = os.path.join(os.path.dirname(os.path.abspath(tiny.__file__)), "checkpoints", "stage2dash2")
HF_REPO1 = "yassermakram/fanous-stage2dash-attn-only-1l"
HF_REPO2 = "yassermakram/fanous-stage2dash2-attn-only-2l"

EVAL_TEXT = (
    "القطة بتاكل السمك والولد بيشرب اللبن في البيت. "
    "الجو حلو النهارده واحنا رايحين نتمشى في الشارع. "
    "المدينة كبيرة وفيها ناس كتير بتروح وتيجي كل يوم. "
    "هو قال إنه هيسافر بكرة بدري عشان الشغل المهم."
)

def _model_from_ckpt(ckpt):
    c = ckpt["config"]
    model = tiny.make_tiny_model(
        n_layers=c["n_layers"], n_heads=c["n_heads"], d_vocab=c["d_vocab"],
        n_ctx=c["n_ctx"], d_model=c["d_model"], attn_only=c["attn_only"],
        normalization_type=c["normalization_type"],
        positional_embedding_type=c["positional_embedding_type"])
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    return model

def _load_real(ckpt_dir, hf_repo):
    from tokenizers import Tokenizer
    tpath, mpath = os.path.join(ckpt_dir, "tokenizer.json"), os.path.join(ckpt_dir, "model.pt")
    if not (os.path.exists(tpath) and os.path.exists(mpath)):
        from huggingface_hub import hf_hub_download
        tpath = hf_hub_download(hf_repo, "tokenizer.json")
        mpath = hf_hub_download(hf_repo, "model.pt")
    ckpt = torch.load(mpath, map_location=tiny.device(), weights_only=False)
    return _model_from_ckpt(ckpt), Tokenizer.from_file(tpath)

def _build_tiny(n_layers):
    from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.normalizer = normalizers.NFKC()
    tok.pre_tokenizer = pre_tokenizers.Whitespace()
    tok.train_from_iterator([EVAL_TEXT] * 20,
                            trainers.BpeTrainer(vocab_size=200, special_tokens=["[UNK]"]))
    model = tiny.make_tiny_model(n_layers=n_layers, n_heads=4, d_vocab=tok.get_vocab_size(),
        n_ctx=64, d_model=64, attn_only=True,
        normalization_type=None, positional_embedding_type="shortformer")
    model.eval()
    return model, tok

if globals().get("FORCE_TINY"):
    model, tok = _build_tiny(n_layers=2)
    model1, _ = _build_tiny(n_layers=1)
else:
    model, tok = _load_real(CKPT2_DIR, HF_REPO2)
    model1, _ = _load_real(CKPT1_DIR, HF_REPO1)

VOCAB = tok.get_vocab_size()
id_to_str = {i: (tok.id_to_token(i) or "[?]") for i in range(VOCAB)}
def encode(text):
    return tok.encode(text).ids

print(f"2-layer: layers={model.cfg.n_layers} heads={model.cfg.n_heads} "
      f"d_model={model.cfg.d_model} n_ctx={model.cfg.n_ctx} vocab={VOCAB}")
print(f"1-layer (2dash): layers={model1.cfg.n_layers}")

<div dir="rtl">

## تجربة: الموديل بطبقة واحدة يفشل في التتابع المتكرر

بنبني تتابع من توكنز حقيقي (من النص العربي فوق)، بعدين بنكرر نفس التتابع. عند كل موضع في النسخة التانية، نقيس **ترتيب التوكن الصح** في توزيع الموديل — رقم أقل = تنبؤ أحسن. الموديل بطبقتين لازم يتفوق هنا.

</div>

## Experiment: the 1-layer model fails on a repeated sequence

We build a sequence from real in-vocab tokens (from the Arabic text above), then repeat it. At each position in the second copy, we measure the **rank of the correct next token** in the model's distribution — lower is better. The 2-layer model should win clearly here.

In [ ]:
def repeat_ids(n):
    base = encode(EVAL_TEXT)[:n]
    seq = base + base
    return torch.tensor([seq]).to(tiny.device()), len(base)

ids2, L = repeat_ids(min(20, model.cfg.n_ctx // 2, model1.cfg.n_ctx // 2))
# at the last position of the first copy's continuation, does the model predict
# the token that actually repeats next?
def rank_of_next(m, ids, L):
    logits = m(ids, return_type="logits")[0]            # ← (pos, V)
    correct = ids[0, L + 1 : 2 * L]                      # the repeated continuation
    preds = logits[L : 2 * L - 1]                        # predictions at those positions
    ranks = (preds.argsort(dim=-1, descending=True) == correct[:, None]).float().argmax(dim=-1)
    return float(ranks.float().mean())

print("mean rank of the repeated token  — 1-layer:", round(rank_of_next(model1, ids2, L), 1))
print("mean rank of the repeated token  — 2-layer:", round(rank_of_next(model,  ids2, L), 1))
# lower is better; the 2-layer model should rank the repeated token far higher.

<div dir="rtl">

## ٢. التفكيك لطبقتين · 2. The two-layer path expansion

لما يكون عندنا طبقتين، الـ logits بتتفكّك لأكتر من مجرد bigram + skip-trigram. المسارات الكاملة هي:

$$\text{logits} = \underbrace{x_0 W_U}_{\text{مباشر (bigram)}} + \underbrace{\sum_{h_0} A^{(0)}_{h_0}\, x_0 W^{(0)}_{V,h_0} W^{(0)}_{O,h_0} W_U}_{\text{رؤوس الطبقة 0 (skip-trigram)}} + \underbrace{\sum_{h_1} A^{(1)}_{h_1}\, x_0 W^{(1)}_{V,h_1} W^{(1)}_{O,h_1} W_U}_{\text{رؤوس الطبقة 1 — مصدرها التضمين}} + \underbrace{\sum_{h_0,h_1} A^{(1)}_{h_1}\, (A^{(0)}_{h_0}\, x_0 W^{(0)}_{V,h_0} W^{(0)}_{O,h_0})\, W^{(1)}_{V,h_1} W^{(1)}_{O,h_1} W_U}_{\text{الرؤوس الافتراضية — التركيب}}$$

المصطلح الأخير هو **التركيب**: رأس في الطبقة 1 بيقرأ مخرج رأس من الطبقة 0، مش التضمين المباشر. الكائن المركّب ده هو اللي بيعمل الاستقراء.

</div>

## 2. The two-layer path expansion

With two layers, the logits expand into more than just bigram + skip-trigram. The full paths are:

$$\text{logits} = \underbrace{x_0 W_U}_{\text{direct (bigram)}} + \underbrace{\sum_{h_0} A^{(0)}_{h_0}\, x_0 W^{(0)}_{V,h_0} W^{(0)}_{O,h_0} W_U}_{\text{layer-0 heads (skip-trigram)}} + \underbrace{\sum_{h_1} A^{(1)}_{h_1}\, x_0 W^{(1)}_{V,h_1} W^{(1)}_{O,h_1} W_U}_{\text{layer-1 heads sourced from embedding}} + \underbrace{\sum_{h_0,h_1} A^{(1)}_{h_1}\, (A^{(0)}_{h_0}\, x_0 W^{(0)}_{V,h_0} W^{(0)}_{O,h_0})\, W^{(1)}_{V,h_1} W^{(1)}_{O,h_1} W_U}_{\text{virtual heads — composition}}$$

The last term is **composition**: a head in layer 1 reads the output of a head in layer 0 rather than the raw embedding. This composed object is what performs induction.

<div dir="rtl">

## ٣. نحمّل الموديل · 3. Load the 2-layer model

الموديل اتحمّل بالفعل في سيل الإعداد فوق (محلي من نقطة حفظ → Hugging Face Hub كاحتياط، أو FORCE_TINY للـ CI). الكونفيغ المطبوع فوق بيوضح: طبقتين انتباه، LN-free، shortformer.

</div>

## 3. Load the 2-layer model

The model was already loaded in the setup cell above (local checkpoint → HF Hub fallback, or FORCE_TINY for CI). The config printed above confirms: two attention layers, LN-free, shortformer positional embeddings.

In [ ]:
ids = torch.tensor([encode(EVAL_TEXT)[: model.cfg.n_ctx]]).to(tiny.device())
logits, cache = model.run_with_cache(ids)
print("logits:", tuple(logits.shape))                 # ← (1, ctx, V)
print("layer-0 attn pattern:", tuple(cache["pattern", 0].shape))  # ← (1, heads, ctx, ctx)
print("layer-1 attn pattern:", tuple(cache["pattern", 1].shape))  # ← (1, heads, ctx, ctx)

<div dir="rtl">

## ٤. التركيب: Q · K · V · 4. Composition: Q / K / V

في النموذج بطبقتين، رأس في الطبقة 1 ممكن "يقرأ" مخرج رأس من الطبقة 0.
الطريقة اللي بيحصل بيها ده هي عبر تركيب المصفوفات. عندنا تلاتة أنواع من التركيب:

- **Q-composition**: رأس في الطبقة 1 بيحسب queries من مخرج الطبقة 0 بدل من التضمين المباشر.
- **K-composition**: بيحسب keys من مخرج الطبقة 0.
- **V-composition**: بيحسب values من مخرج الطبقة 0.

نقيس قوة التركيب بنسبة فروبينيوس: `||A B|| / (||A|| ||B||)`. القيمة العالية تعني تركيب أقوى.
التلاتة أنواع موجودين دايمًا، بس مش كلهم بيعملوا الاستقراء — سنوضّح ده في §٨.

</div>

## 4. Composition: Q / K / V

In a two-layer model, a head in layer 1 can "read" the output of a head in layer 0.
This happens through weight-space composition. There are three composition types:

- **Q-composition**: layer-1 head computes queries from layer-0 output instead of the raw embedding.
- **K-composition**: computes keys from layer-0 output.
- **V-composition**: computes values from layer-0 output.

We measure composition strength with a Frobenius-norm ratio: `||A B|| / (||A|| ||B||)`. A high score means stronger composition.
All three types exist, but not all build induction — we will clarify this in §8.

In [ ]:
import torch

def fro(x):
    return float(torch.linalg.matrix_norm(x, ord="fro"))

# Layer-0 output writes W_OV0 = W_V0 @ W_O0 into the residual; layer-1 heads read it
# through their W_Q1 / W_K1 / W_V1. Composition score = ||A B|| / (||A|| ||B||).
W_OV0 = torch.stack([model.W_V[0, h] @ model.W_O[0, h] for h in range(model.cfg.n_heads)])  # (H, d, d)

def comp_scores(W_in1):  # W_in1: (H, d_model, d_head) for layer-1 Q/K/V
    out = torch.zeros(model.cfg.n_heads, model.cfg.n_heads)
    for h0 in range(model.cfg.n_heads):
        for h1 in range(model.cfg.n_heads):
            prod = W_OV0[h0] @ W_in1[h1]
            out[h0, h1] = fro(prod) / (fro(W_OV0[h0]) * fro(W_in1[h1]) + 1e-9)
    return out.detach()

q_comp = comp_scores(model.W_Q[1])
k_comp = comp_scores(model.W_K[1])
v_comp = comp_scores(model.W_V[1])
print("Q-composition (layer0 head -> layer1 head):", tuple(q_comp.shape))  # ← (H, H)
print("max K-composition:", round(float(k_comp.max()), 3),
      "at", [int(x) for x in divmod(int(k_comp.argmax()), k_comp.shape[1])])

<div dir="rtl">

## ٥. الرؤوس الافتراضية · 5. Virtual attention heads

كل زوج (رأس طبقة 0 → رأس طبقة 1) مع K-composition قوية بيعمل مع بعض كـ "رأس افتراضي" واحد.
الرأس الافتراضي ده بيطبّق attention بتتحدد بدرجتين دفعة واحدة:
الدرجة الأولى في الطبقة 0 بتحدد "فين أكتب"، والدرجة التانية في الطبقة 1 بتحدد "فين أقرأ".
مصفوفة K-composition هي خريطة قوة الرؤوس الافتراضية دي.

</div>

## 5. Virtual attention heads

Each (layer-0 head → layer-1 head) pair with strong K-composition acts together as one "virtual head".
This virtual head applies attention governed by two degrees at once:
the first degree in layer 0 determines "where to write", and the second in layer 1 determines "where to read".
The K-composition matrix is exactly the map of these virtual-head strengths.

In [ ]:
fig = go.Figure(data=go.Heatmap(
    z=k_comp.numpy(),
    x=[f"L1 h{h}" for h in range(model.cfg.n_heads)],
    y=[f"L0 h{h}" for h in range(model.cfg.n_heads)],
    colorscale="Blues"))
fig.update_layout(title="K-composition / virtual-head strength", height=380)
fig.show()

<div dir="rtl">

## ٦. أهمية الحدود · 6. Term importance

في التفكيك الكامل للـ logits، كل مصطلح (المسار المباشر، كل رأس منفرد، كل رأس افتراضي)
بيضيف حاجة للتنبؤ. بنقيس أهمية كل مصطلح بنورم فروبينيوس لمصفوفة تأثيره على الـ logits: `||W_E OV W_U||`.
ترتيب الحدود بيوضّح إيه اللي بيسيطر على تنبؤات الموديل.

</div>

## 6. Term importance

In the full logit decomposition, every term (direct path, each individual head, each virtual head)
contributes to the prediction. We measure each term's importance by the Frobenius norm of its
logit-contribution matrix: `||W_E OV W_U||`.
Ranking the terms reveals what dominates the model's predictions.

In [ ]:
terms = {"direct (W_E W_U)": fro(model.W_E @ model.W_U)}
for L in range(model.cfg.n_layers):
    for h in range(model.cfg.n_heads):
        ov = model.W_V[L, h] @ model.W_O[L, h]
        terms[f"L{L} h{h}"] = fro(model.W_E @ ov @ model.W_U)
top = sorted(terms.items(), key=lambda kv: kv[1], reverse=True)[:8]
for name, val in top:
    print(f"  {name:>16}: {val:,.1f}")

<div dir="rtl">

## ٧. تحليل القيم الذاتية للنسخ · 7. Eigenvalue copying analysis

دائرة OV اللي بتنسخ بتكون ليها قيم ذاتية حقيقية موجبة في المقام الأكبر.
ده لأن النسخ التامة تعادل مصفوفة هوية، واللي كل قيمها الذاتية موجبة (+1).
بنحسب القيم الذاتية لـ `W_O W_V` (المسقط من فضاء الرأس للـ residual) كوكيل تطبيقي
عن تحليل الورقة الأصلية، ونقيس نسبة القيم الذاتية الحقيقية الموجبة لكل رأس.

</div>

## 7. Eigenvalue copying analysis

A copying OV circuit has mostly-positive real eigenvalues.
This is because perfect copying corresponds to an identity matrix, all of whose eigenvalues are +1.
We compute eigenvalues of `W_O W_V` (the projection from head space to the residual) as a tractable
proxy for the paper's full analysis, and report the positive-eigenvalue fraction per head.
Heads with a high fraction are "copying" heads — good candidates for induction head component roles.

In [ ]:
def copying_fraction(L, h):
    ov = (model.W_V[L, h] @ model.W_O[L, h]).detach()      # (d_model, d_model)
    eig = torch.linalg.eigvals(ov).real
    return float((eig > 0).float().mean())

ov_eig = {(L, h): copying_fraction(L, h)
          for L in range(model.cfg.n_layers) for h in range(model.cfg.n_heads)}
for (L, h), frac in sorted(ov_eig.items(), key=lambda kv: kv[1], reverse=True)[:6]:
    print(f"  L{L} h{h}: positive-eigenvalue fraction = {frac:.2f}")